# 05 · Portfolio evaluation protocol

Create reproducible grouped nested assignments **without fitting models**. This is exploratory internal evaluation: original patient identities are unavailable and earlier test outcomes were inspected. No new split reverses that exposure. [Written protocol](../docs/portfolio-evaluation-protocol.md).

Run after Notebooks 00–02. Set `APNEA_PORTFOLIO_RUN` to a new directory. Synthetic offline smoke mode: `APNEA_PORTFOLIO_SMOKE=1`. Baselines and compute eligibility remain pending; this protocol stays **draft** until Plans 002–003 are complete.

In [ ]:
import hashlib
import importlib.metadata
import json
import os
import platform
import tempfile
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display
from sklearn.metrics import (
    accuracy_score,
    average_precision_score,
    confusion_matrix,
    f1_score,
    precision_recall_curve,
    precision_score,
    roc_auc_score,
)
from sklearn.model_selection import StratifiedGroupKFold

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())
SMOKE = os.environ.get("APNEA_PORTFOLIO_SMOKE") == "1"


def resolve_path(value):
    path = Path(value)
    return path if path.is_absolute() else ROOT / path


RUN = resolve_path(
    os.environ.get("APNEA_PORTFOLIO_RUN", "data/processed/portfolio-v2/protocol-20261108")
)
PHASE1 = resolve_path(os.environ.get("APNEA_RUN", "data/processed/phase1"))
GROUPS = resolve_path(os.environ.get("APNEA_SPLIT_RUN", "data/processed/phase1-grouped"))
FEATURE_RUN = resolve_path(os.environ.get("APNEA_PHASE2_RUN", "data/processed/phase2-grouped"))
print("Run:", RUN, "Synthetic:", SMOKE)

## 1. Construct and independently validate the nested assignments

Five outer folds (seed 67); three inner folds (seed 167 + outer fold); four CNN stopping subfolds (seed 267 + 10 × outer + inner), with subfold zero used for stopping. All assignment construction uses labels for stratification and the saved literature-derived group identity; no feature values or model scores select splits. Validation regenerates the deterministic contract and checks class support.

In [ ]:
def validate_cohort(cohort):
    required = {
        "record_id",
        "minute",
        "subject_group",
        "label",
        "start_sample",
        "end_sample",
        "quality_flag",
    }
    if not required.issubset(cohort) or cohort.empty or cohort[list(required)].isna().any().any():
        raise ValueError("Missing or empty cohort fields")
    if cohort.duplicated(["record_id", "minute"]).any() or set(cohort.label) != {0, 1}:
        raise ValueError("Duplicate keys or nonbinary/class-degenerate cohort")
    if cohort.groupby("record_id").subject_group.nunique().max() != 1:
        raise ValueError("A recording has conflicting group identities")
    if not (
        (cohort.end_sample - cohort.start_sample).eq(6000).all()
        and cohort.start_sample.eq(cohort.minute * 6000).all()
    ):
        raise ValueError("Invalid minute bounds")


def require_classes(table):
    if set(table.label) != {0, 1}:
        raise ValueError("Class-infeasible fitting, stopping or scoring partition")


def build_fold_assignments(
    cohort, outer_seed=67, outer_splits=5, inner_splits=3, stopping_splits=4
):
    validate_cohort(cohort)
    t = cohort.sort_values(["record_id", "minute"]).reset_index(drop=True)
    assignments, stops = [], []
    cv = StratifiedGroupKFold(outer_splits, shuffle=True, random_state=outer_seed)
    for outer, (fit, held) in enumerate(cv.split(t, t.label, t.subject_group)):
        train, test = t.iloc[fit].reset_index(drop=True), t.iloc[held]
        require_classes(train)
        require_classes(test)
        records = t[["record_id", "subject_group"]].drop_duplicates().copy()
        records["outer_fold"] = outer
        records["outer_role"] = np.where(records.record_id.isin(test.record_id), "test", "train")
        records["inner_fold"] = -1
        inner_cv = StratifiedGroupKFold(inner_splits, shuffle=True, random_state=167 + outer)
        for inner, (inner_fit, inner_held) in enumerate(
            inner_cv.split(train, train.label, train.subject_group)
        ):
            learning, validation = (
                train.iloc[inner_fit].reset_index(drop=True),
                train.iloc[inner_held],
            )
            require_classes(learning)
            require_classes(validation)
            records.loc[records.record_id.isin(validation.record_id), "inner_fold"] = inner
            stop_cv = StratifiedGroupKFold(
                stopping_splits, shuffle=True, random_state=267 + 10 * outer + inner
            )
            calibration_fit, calibration_stop = next(
                stop_cv.split(learning, learning.label, learning.subject_group)
            )
            require_classes(learning.iloc[calibration_fit])
            require_classes(learning.iloc[calibration_stop])
            for role, part in [
                ("calibration-fit", learning.iloc[calibration_fit]),
                ("calibration-stop", learning.iloc[calibration_stop]),
                ("inner-validation", validation),
            ]:
                row = part[["record_id", "subject_group"]].drop_duplicates().copy()
                row["outer_fold"], row["inner_fold"], row["role"] = outer, inner, role
                stops.append(row)
        assignments.append(records)
    return (pd.concat(assignments, ignore_index=True), pd.concat(stops, ignore_index=True))


def validate_fold_assignments(cohort, assignments, stop_assignments, **settings):
    expected, expected_stops = build_fold_assignments(cohort, **settings)
    for actual, target, keys in [
        (assignments, expected, ["outer_fold", "record_id"]),
        (stop_assignments, expected_stops, ["outer_fold", "inner_fold", "record_id"]),
    ]:
        if set(actual.columns) != set(target.columns) or actual.duplicated(keys).any():
            raise ValueError("Invalid assignment schema or duplicate assignments")
        try:
            pd.testing.assert_frame_equal(
                actual[target.columns].sort_values(keys).reset_index(drop=True),
                target.sort_values(keys).reset_index(drop=True),
                check_dtype=False,
            )
        except AssertionError as exc:
            raise ValueError("Assignments differ from deterministic grouped contract") from exc
    if assignments.query("outer_role == 'test'").groupby("record_id").size().ne(1).any():
        raise ValueError("Incomplete outer test coverage")

## 2. Define metrics before model selection

Select candidates by mean inner average precision (ties within 1e-12 keep earlier candidates); select thresholds from inner out-of-fold scores by maximum F1, lowest threshold on ties. Ranking scores may be uncalibrated. Never choose thresholds from outer predictions.

The primary summary is mean outer-fold AP; pooled out-of-fold metrics are separate estimands. One-class groups have missing AP/ROC-AUC with an explicit reason. Fold standard deviation is descriptive, not a confidence interval.

In [ ]:
def choose_threshold(labels, scores):
    labels, scores = np.asarray(labels), np.asarray(scores, dtype=float)
    if labels.shape != scores.shape or set(labels) != {0, 1} or not np.isfinite(scores).all():
        raise ValueError("Invalid threshold-selection inputs")
    precision, recall, thresholds = precision_recall_curve(labels, scores)
    f1 = 2 * precision[:-1] * recall[:-1] / np.maximum(precision[:-1] + recall[:-1], 1e-12)
    return float(thresholds[np.argmax(f1)])


def validate_predictions(table):
    required = [
        "record_id",
        "minute",
        "subject_group",
        "label",
        "score",
        "prediction",
        "outer_fold",
    ]
    if not set(required).issubset(table) or table.empty or table[required].isna().any().any():
        raise ValueError("Incomplete predictions")
    if table.duplicated(["record_id", "minute"]).any():
        raise ValueError("Predictions must describe one method/configuration/seed with unique keys")
    if not set(table.label).issubset({0, 1}) or not set(table.prediction).issubset({0, 1}):
        raise ValueError("Nonbinary labels/predictions")
    if not np.isfinite(table.score).all():
        raise ValueError("Nonfinite scores")
    if table.groupby("subject_group").outer_fold.nunique().max() != 1:
        raise ValueError("A group crosses outer test folds")


def metric_values(table, weights=None):
    y, prediction = table.label.to_numpy(), table.prediction.to_numpy()
    weights = np.ones(len(table)) if weights is None else np.asarray(weights)
    active = weights > 0
    both = len(np.unique(y[active])) == 2
    tn, fp, fn, tp = confusion_matrix(y, prediction, labels=[0, 1], sample_weight=weights).ravel()
    return dict(
        n_windows=float(weights.sum()),
        prevalence=float(np.average(y, weights=weights)),
        tn=float(tn),
        fp=float(fp),
        fn=float(fn),
        tp=float(tp),
        sensitivity=float(tp / (tp + fn)) if tp + fn else np.nan,
        specificity=float(tn / (tn + fp)) if tn + fp else np.nan,
        precision=float(precision_score(y, prediction, sample_weight=weights, zero_division=0)),
        f1_score=float(f1_score(y, prediction, sample_weight=weights, zero_division=0)),
        accuracy=float(accuracy_score(y, prediction, sample_weight=weights)),
        average_precision=float(average_precision_score(y, table.score, sample_weight=weights))
        if both
        else np.nan,
        roc_auc=float(roc_auc_score(y, table.score, sample_weight=weights)) if both else np.nan,
        ranking_reason=None if both else "single_class",
    )


def summarize_predictions(table):
    validate_predictions(table)
    rows = [dict(level="pooled", identifier="all", **metric_values(table))]
    for level, column in [
        ("fold", "outer_fold"),
        ("record", "record_id"),
        ("group", "subject_group"),
    ]:
        for key, part in table.groupby(column, sort=True):
            rows.append(dict(level=level, identifier=str(key), **metric_values(part)))
    folds = [r["average_precision"] for r in rows if r["level"] == "fold"]
    rows.append(
        dict(
            level="fold_mean",
            identifier="primary",
            average_precision=float(np.mean(folds)),
            fold_ap_std=float(np.std(folds, ddof=1)) if len(folds) > 1 else np.nan,
        )
    )
    return pd.DataFrame(rows)


def group_weights(table, sampled_groups):
    counts = pd.Series(sampled_groups).value_counts()
    return table.subject_group.map(counts).fillna(0).to_numpy(dtype=float)


def paired_group_bootstrap(
    left, right, metric="average_precision", n_resamples=2000, seed=20261108
):
    validate_predictions(left)
    validate_predictions(right)
    keys = ["record_id", "minute"]
    a, b = (t.sort_values(keys).reset_index(drop=True) for t in (left, right))
    if not a[keys + ["subject_group", "label", "outer_fold"]].equals(
        b[keys + ["subject_group", "label", "outer_fold"]]
    ):
        raise ValueError("Unpaired keys, labels, groups or outer folds")
    groups = np.sort(a.subject_group.unique())
    rng, differences = np.random.default_rng(seed), []
    for _ in range(n_resamples):
        weights = group_weights(a, rng.choice(groups, size=len(groups), replace=True))
        if len(np.unique(a.label.to_numpy()[weights > 0])) < 2:
            continue
        differences.append(metric_values(a, weights)[metric] - metric_values(b, weights)[metric])
    valid = len(differences)
    interval = np.percentile(differences, [2.5, 97.5]) if valid >= 1000 else [np.nan, np.nan]
    return dict(
        metric=metric,
        estimate=metric_values(a)[metric] - metric_values(b)[metric],
        lower=float(interval[0]),
        upper=float(interval[1]),
        valid_draws=valid,
        skipped_draws=n_resamples - valid,
        seed=seed,
        scope="exploratory; fixed predictions only; excludes tuning uncertainty",
    )

## 3. Save a draft protocol atomically and refuse incompatible reuse

Source, environment, configuration and output hashes protect run reuse. Synthetic outputs are explicitly marked and never satisfy real-cohort checks. A matching existing run is verified, not overwritten. Baseline eligibility and timing are pending; there is no model training or outer-score inspection here.

In [ ]:
def file_hash(path):
    return hashlib.sha256(Path(path).read_bytes()).hexdigest()


def save_protocol(run, cohort, assignments, stops, config, sources):
    run = Path(run)
    environment = {
        name: importlib.metadata.version(name)
        for name in ["numpy", "pandas", "scikit-learn", "pyarrow"]
    }
    identity = dict(
        configuration=config,
        sources=sources,
        environment=environment,
        python=platform.python_version(),
    )
    signature = hashlib.sha256(json.dumps(identity, sort_keys=True).encode()).hexdigest()
    target = run / "protocol"
    if target.exists():
        manifest = json.loads((target / "manifest.json").read_text())
        if manifest["identity_hash"] != signature:
            raise ValueError("Incompatible source/configuration/environment: choose a fresh run")
        if any(
            file_hash(target / name) != digest for name, digest in manifest["artifacts"].items()
        ):
            raise ValueError("Existing protocol artifact hash mismatch")
        return manifest
    run.mkdir(parents=True, exist_ok=True)
    with tempfile.TemporaryDirectory(prefix=".protocol-", dir=run) as temporary:
        stage = Path(temporary) / "protocol"
        stage.mkdir()
        cohort.to_parquet(stage / "cohort.parquet", index=False)
        assignments.to_csv(stage / "fold_assignments.csv", index=False)
        stops.to_csv(stage / "cnn_stop_assignments.csv", index=False)
        (stage / "protocol.json").write_text(json.dumps(config, indent=2) + "\n")
        (stage / "metrics.json").write_text(
            json.dumps(
                dict(
                    primary="mean outer-fold average_precision_score",
                    secondary="pooled OOF metrics; record/group summaries",
                    threshold="maximum inner OOF F1; lowest threshold on ties",
                    bootstrap="group weights; 2000 draws; 2.5/97.5 percentiles; >=1000 valid",
                    seed=20261108,
                    score_kind="probability or decision score; calibration not assumed",
                ),
                indent=2,
            )
            + "\n"
        )
        artifacts = {p.name: file_hash(p) for p in stage.iterdir()}
        manifest = dict(identity_hash=signature, **identity, artifacts=artifacts)
        (stage / "manifest.json").write_text(json.dumps(manifest, indent=2) + "\n")
        stage.rename(target)
    return manifest

## 4. Load the cohort, verify feature alignment, and display partition checks

Labels describe apnea status at the beginning of the associated minute. They do not establish events anywhere in that minute or clinical AHI. Keep quality-flagged and feature-missing windows. Existing group identity is reused; historical split membership is not used as the new evaluation assignment.

In [ ]:
sources = {
    "notebook": file_hash(ROOT / "notebooks/05_evaluation_protocol.ipynb"),
    "lock": file_hash(ROOT / "uv.lock"),
    "written_protocol": file_hash(ROOT / "docs/portfolio-evaluation-protocol.md"),
}
if SMOKE:
    cohort = pd.DataFrame(
        [
            dict(
                record_id=f"synthetic_{g}_{r}",
                minute=m,
                subject_group=f"g{g}",
                label=m % 2,
                start_sample=m * 6000,
                end_sample=(m + 1) * 6000,
                quality_flag=False,
            )
            for g in range(16)
            for r in range(2)
            for m in range(4)
        ]
    )
    settings = dict(outer_seed=67, outer_splits=2, inner_splits=2, stopping_splits=2)
else:
    inputs = {
        "windows": PHASE1 / "windows.parquet",
        "groups": GROUPS / "subject_splits.csv",
        "features": FEATURE_RUN / "features.parquet",
    }
    sources.update({name: file_hash(path) for name, path in inputs.items()})
    windows, groups = pd.read_parquet(inputs["windows"]), pd.read_csv(inputs["groups"])
    cohort = windows[
        ["record_id", "minute", "label", "start_sample", "end_sample", "quality_flag"]
    ].merge(groups[["record_id", "subject_group"]], on="record_id", validate="many_to_one")
    features = pd.read_parquet(inputs["features"])
    key = ["record_id", "minute"]
    pd.testing.assert_frame_equal(
        cohort[key + ["label"]].sort_values(key).reset_index(drop=True),
        features[key + ["label"]].sort_values(key).reset_index(drop=True),
        check_dtype=False,
    )
    if (len(cohort), cohort.record_id.nunique(), cohort.subject_group.nunique()) != (16558, 34, 24):
        raise ValueError("Unexpected real cohort; amend protocol before proceeding")
    settings = dict(outer_seed=67, outer_splits=5, inner_splits=3, stopping_splits=4)
cohort = cohort[
    ["record_id", "minute", "subject_group", "label", "start_sample", "end_sample", "quality_flag"]
]
assignments, stops = build_fold_assignments(cohort, **settings)
validate_fold_assignments(cohort, assignments, stops, **settings)
config = dict(
    version="2.0-draft",
    status="draft",
    synthetic=SMOKE,
    **settings,
    primary_training_seed=67,
    repeat_seeds=[167, 267],
    input_hz=100,
    input_samples=6000,
    baseline_eligibility="pending Plan 002",
    search_compute="pending Plan 003",
    limitations=[
        "original patient identities unavailable",
        "prior test exposure",
        "retrospective research only",
    ],
)
manifest = save_protocol(RUN, cohort, assignments, stops, config, sources)
partitioned = cohort.merge(assignments, on=["record_id", "subject_group"], validate="many_to_many")


def partition_counts(table, columns):
    return (
        table.groupby(columns)
        .agg(
            groups=("subject_group", "nunique"),
            recordings=("record_id", "nunique"),
            minutes=("label", "size"),
            positives=("label", "sum"),
        )
        .assign(negatives=lambda x: x.minutes - x.positives)
    )


display(partition_counts(partitioned, ["outer_fold", "outer_role"]))
stopping = cohort.merge(stops, on=["record_id", "subject_group"], validate="many_to_many")
display(partition_counts(stopping, ["outer_fold", "inner_fold", "role"]))
print("Protocol identity:", manifest["identity_hash"], "Status: draft; no models fitted")

## Interpretation and next step

The checks establish assignment consistency and class feasibility, not verified patient independence or clinical generalization. The primary RQ1 ablation will keep Phase 2 settings fixed across feature sets; RQ2 will tune full F2 pipelines versus the unchanged compact CNN architecture at 100 Hz. Literature eligibility, candidate definitions and timing must be completed before freezing this draft. See Plans 002–003. Source notebooks remain output-free; executed copies belong under ignored reports/.